# Eclat

ECLAT is an unsupervised frequent-itemset mining algorithm that discovers items frequently purchased together by using a vertical transaction format and set intersection.

# For more info click the link below

https://docs.google.com/document/d/1E7EK5wQNecSnO_jpSzJIrs4a9RjLe-x-MG1ZDDhokKA/edit?usp=sharing

# Notes and interview Q and answer

https://docs.google.com/document/d/1j0KqRhnTuImlXQsrR3rRlOqmXNtilS0nviBECRPq1Rw/edit?usp=sharing

# Coding

# Dataset 

`We will reuse the same supermarket transaction because thsi is directly connected to FP-Growth and association Rule learning.`

# import libraries 

In [1]:
import pandas as pd   
from itertools import combinations                                   # Data manipulation

 # Create  transactions 

In [2]:
transactions = [
    ["Milk", "Bread", "Butter"],                         # Transaction 1
    ["Milk", "Bread"],                                   # Transaction 2
    ["Milk", "Bread", "Butter", "Eggs"],                 # Transaction 3
    ["Bread", "Butter"],                                 # Transaction 4
    ["Milk", "Bread", "Eggs"],                           # Transaction 5
    ["Milk", "Butter"],                                  # Transaction 6
    ["Bread", "Butter", "Eggs"],                         # Transaction 7
    ["Milk", "Bread", "Butter"],                         # Transaction 8
    ["Milk", "Bread", "Eggs"],                           # Transaction 9
    ["Bread", "Butter"]                                  # Transaction 10
]

# Convert transaction into vertical format 

`This is the most important concept in ECLAT`

Normal/horizontal representation

Transaction 1 → Milk, Bread, Butter

Transaction 2 → Milk, Bread

Transaction 3 → Milk, Bread, Butter, Eggs


Eclat Changes it to: 

Milk   → {1, 2, 3, 5, 6, 8, 9}

Bread  → {1, 2, 3, 4, 5, 7, 8, 9, 10}

Butter → {1, 3, 4, 6, 7, 8, 10}

Eggs   → {3, 5, 7, 9}






THese are called TID sets. 

TID = Transaction ID

In [4]:
tid_sets = {}                                            # Store transaction IDs for every item


for tid, transaction in enumerate(
    transactions,                                        # Loop through every transaction
    start=1                                              # Start transaction IDs from 1
):
    for item in transaction:                             # Loop through items in each transaction

        if item not in tid_sets:
            tid_sets[item] = set()                       # Create an empty TID set for a new item

        tid_sets[item].add(
            tid                                                # Add transaction ID containing this item
        )


print(tid_sets)

{'Milk': {1, 2, 3, 5, 6, 8, 9}, 'Bread': {1, 2, 3, 4, 5, 7, 8, 9, 10}, 'Butter': {1, 3, 4, 6, 7, 8, 10}, 'Eggs': {9, 3, 5, 7}}


# Calculate support 

`Here is the beautiful part of ECLAT.`

`For one item`

Milk
→ {1,2,3,5,6,8,9}

Number of transactions = 7

Support = 7 / 10
        = 0.70

`In code `

In [6]:
# This operation is the heart of the ECLAT.


min_support = 0.3                                      # Require at least 30% transaction support

n_transactions = len(
    transactions                                          # Total number of transactions
)


for item, tids in tid_sets.items():

    support = len(
        tids                                                # Number of transactions containing the item
    ) / n_transactions

    if support >= min_support:
        print(
            item,
            "→ support:",
            support
        )

Milk → support: 0.7
Bread → support: 0.9
Butter → support: 0.7
Eggs → support: 0.4


In [8]:
# Let's create them: 

tid_sets = {}                                            # Store transaction IDs for every item


for tid, transaction in enumerate(
    transactions,                                        # Loop through every transaction
    start=1                                              # Start transaction IDs from 1
):
    for item in transaction:                             # Loop through items in each transaction

        if item not in tid_sets:
            tid_sets[item] = set()                       # Create an empty TID set for a new item

        tid_sets[item].add(
            tid                                                # Add transaction ID containing this item
        )


print(tid_sets)

# This interection operation is the heart of ECLAT 

{'Milk': {1, 2, 3, 5, 6, 8, 9}, 'Bread': {1, 2, 3, 4, 5, 7, 8, 9, 10}, 'Butter': {1, 3, 4, 6, 7, 8, 10}, 'Eggs': {9, 3, 5, 7}}


In [9]:
# FInd frequent 2 items 

# let's automate that : 

from itertools import combinations                         # Generate combinations of items


frequent_itemsets = []                                    # Store discovered frequent itemsets


items = list(
    tid_sets.keys()                                       # Get all unique items
)


for item1, item2 in combinations(
    items,
    2                                                     # Generate pairs of items
):

    common_tids = tid_sets[item1].intersection(
        tid_sets[item2]                                   # Find transactions containing both items
    )

    support = len(
        common_tids
    ) / n_transactions                                    # Calculate support


    if support >= min_support:

        frequent_itemsets.append(
            (
                (item1, item2),                           # Store the itemset
                support                                   # Store its support
            )
        )


print(frequent_itemsets)

[(('Milk', 'Bread'), 0.6), (('Milk', 'Butter'), 0.4), (('Milk', 'Eggs'), 0.3), (('Bread', 'Butter'), 0.6), (('Bread', 'Eggs'), 0.4)]


# Find the larger itemsets 

`ECLAT can continue the same process`

In [10]:
milk_bread = (
    tid_sets["Milk"]
    .intersection(tid_sets["Bread"])                     # Transactions containing Milk + Bread
)

milk_bread_butter = milk_bread.intersection(
    tid_sets["Butter"]                                   # Add Butter to the itemset
)

print(
    "Milk + Bread + Butter:",
    milk_bread_butter
)

print(
    "Support:",
    len(milk_bread_butter) / n_transactions               # Calculate support
)

Milk + Bread + Butter: {8, 1, 3}
Support: 0.3


# A simple ECLAT implementation 

`Now let's make a reusable function.`

In [11]:
def eclat(
    tid_sets,                                             # Vertical transaction representation
    n_transactions,                                      # Total number of transactions
    min_support                                           # Minimum support threshold
):

    frequent_itemsets = []                               # Store all frequent itemsets


    def search(
        prefix,                                           # Current itemset
        items_and_tids                                    # Remaining items and their TID sets
    ):

        for i in range(
            len(items_and_tids)                           # Loop through remaining items
        ):

            item, tids = items_and_tids[i]                # Get current item and its TID set


            support = len(
                tids
            ) / n_transactions                            # Calculate support


            if support >= min_support:

                new_itemset = prefix + (
                    item,
                )                                         # Add current item to the itemset


                frequent_itemsets.append(
                    (
                        new_itemset,
                        support
                    )
                )


                new_items = []                            # Store items for the next recursion


                for next_item, next_tids in items_and_tids[
                    i + 1:
                ]:

                    intersection = tids.intersection(
                        next_tids                            # Intersect TID sets
                    )


                    if (
                        len(intersection) / n_transactions
                        >= min_support
                    ):

                        new_items.append(
                            (
                                next_item,
                                intersection
                            )
                        )


                search(
                    new_itemset,                           # Continue growing the itemset
                    new_items                              # Continue with intersected TID sets
                )


    initial_items = list(
        tid_sets.items()                                  # Convert vertical data into item-TID pairs
    )


    search(
        tuple(),                                           # Start with an empty itemset
        initial_items                                      # Start searching all items
    )


    return frequent_itemsets                              # Return all frequent itemsets

In [12]:
results = eclat(
    tid_sets,                                             # Vertical transaction data
    n_transactions,                                      # Number of transactions
    min_support=0.3                                      # Minimum support of 30%
)


for itemset, support in results:

    print(
        itemset,
        "→ support:",
        support
    )

('Milk',) → support: 0.7
('Milk', 'Bread') → support: 0.6
('Milk', 'Bread', 'Butter') → support: 0.3
('Milk', 'Bread', 'Eggs') → support: 0.3
('Milk', 'Butter') → support: 0.4
('Milk', 'Eggs') → support: 0.3
('Bread',) → support: 0.9
('Bread', 'Butter') → support: 0.6
('Bread', 'Eggs') → support: 0.4
('Butter',) → support: 0.7
('Eggs',) → support: 0.4


# Experiment with min_support
`This is the main hyperparameter to experiment with`

In [13]:
for support in [0.2, 0.3, 0.4, 0.5]:

    results = eclat(
        tid_sets,                                         # Vertical transaction data
        n_transactions,                                  # Total transactions
        min_support=support                              # Change minimum support
    )

    print(
        f"min_support={support} → "
        f"{len(results)} frequent itemsets"
    )

min_support=0.2 → 13 frequent itemsets
min_support=0.3 → 11 frequent itemsets
min_support=0.4 → 8 frequent itemsets
min_support=0.5 → 5 frequent itemsets
